# PA3: Search over Reasoning
Kaggle T4 x2 pipeline: Qwen2.5-1.5B generator, Qwen2.5-7B (NF4) step judge, GSM8K-50. Designed for unattended Save & Run All execution; all artifacts are written under `/kaggle/working`.

## Setup

In [ ]:
import glob
import os
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/syeddaniyalg/cs5326-pa3.git"
REPO = "/kaggle/working/cs5326-pa3"

if os.path.isdir(os.path.join(REPO, ".git")):
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, REPO], check=True)
os.chdir(REPO)
sys.path.insert(0, REPO)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "bitsandbytes"], check=True)

restored = []
for source in glob.glob("/kaggle/input/**/checkpoints", recursive=True):
    target = os.path.join(REPO, "outputs", "checkpoints")
    if os.path.isfile(os.path.join(source, "generation.jsonl")) and not os.path.exists(target):
        shutil.copytree(source, target)
        restored.append(source)
print("restored checkpoints from:", restored or "none")

In [ ]:
import torch

print("torch", torch.__version__, "| cuda", torch.version.cuda)
for index in range(torch.cuda.device_count()):
    properties = torch.cuda.get_device_properties(index)
    print(f"cuda:{index}", properties.name, f"{properties.total_memory / 1024**3:.1f} GB")
assert torch.cuda.device_count() >= 1, "no GPU visible"

## Model download

In [ ]:
from src.pipeline import prefetch_models
from src.utils import GENERATOR_ID, JUDGE_ID

prefetch_models([GENERATOR_ID, JUDGE_ID])

## Pipeline
Generation (greedy, SC-5, SC-15) runs on `cuda:0` while the judge scores the shared 5-path pools on `cuda:1`. Per-problem results are checkpointed to `outputs/checkpoints`, so an interrupted run resumes where it stopped.

In [ ]:
import run_all

try:
    artifacts = run_all.main([])
finally:
    archive = shutil.make_archive("/kaggle/working/pa3_outputs", "zip", os.path.join(REPO, "outputs"))
    print("packaged:", archive)

## Results

In [ ]:
from IPython.display import Image, Markdown, display

display(Markdown(open("outputs/tables/report_tables.md", encoding="utf-8").read()))
for figure in sorted(glob.glob("outputs/figures/*.png")):
    display(Image(figure))